# Interpretability — SHAP, XPER, permutation importance

Computes SHAP attributions, XPER performance decompositions, and permutation
importance for whichever models have a `models/<name>_model.py` file so far (see
`common_metrics.report_status()` below — copy `models/TEMPLATE_model.py` to plug
one in).

**Framework only right now**: every function below is fully implemented against the
real feature schema. Until a real model lands, the last cell runs a throwaway
smoke test to prove the harness works end to end — delete that section once at
least one real model is in.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("../..").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.metrics import roc_auc_score

from common_metrics import (
    FEATURES, PROTECTED_COLS, AUDIT_COLS, MODEL_NAMES, TEAM_THRESHOLD,
    load_split, get_X_y, get_audit, available_models, report_status,
    manual_permutation_importance,
)

report_status()

## Load data

In [ ]:
train_df = load_split("train")
val_df = load_split("val")
test_df = load_split("test")

X_train, y_train = get_X_y(train_df)
X_val, y_val = get_X_y(val_df)
X_test, y_test = get_X_y(test_df)

print(f"train: {X_train.shape}, val: {X_val.shape}, test: {X_test.shape}")

## SHAP — per-prediction attribution

TreeExplainer for XGBoost, LinearExplainer for logreg, KernelExplainer for TabPFN —
with a fallback to KernelExplainer if a model's native explainer can't be built from
whatever `fit()` returned (e.g. if it's wrapped in a pipeline).

In [ ]:
MODEL_EXPLAINER_TYPE = {"xgboost": "tree", "logreg": "linear", "tabpfn": "kernel"}

def get_shap_explainer(model_type, model, module, X_background):
    predict_fn = lambda X: module.predict_proba(model, pd.DataFrame(X, columns=X_background.columns))[:, 1]
    if model_type == "tree":
        try:
            return shap.TreeExplainer(model), "TreeExplainer (exact)"
        except Exception as e:
            print(f"  TreeExplainer failed ({e}) — falling back to KernelExplainer")
    elif model_type == "linear":
        try:
            return shap.LinearExplainer(model, X_background), "LinearExplainer (exact)"
        except Exception as e:
            print(f"  LinearExplainer failed ({e}) — falling back to KernelExplainer")
    background_sample = shap.sample(X_background, min(100, len(X_background)))
    return shap.KernelExplainer(predict_fn, background_sample), "KernelExplainer (approximate)"

## XPER — performance decomposition

`pip install XPER` ([github.com/hi-paris/XPER](https://github.com/hi-paris/XPER)).
Decomposes a performance metric (not a prediction) into per-feature Shapley
contributions. `Eval_Metric` is a fixed metric name (`"AUC"`, `"Accuracy"`, `"MC"`
for misclassification cost, etc.) rather than an arbitrary function — `"MC"` takes
`CFP`/`CFN` (cost of a false positive / false negative) for an economic-cost
decomposition. `kernel=True` is recommended above ~10 features, so it's the default
here given a 32-feature model.

**Runtime scales with feature count, not row count** — the Shapley coalition
sampling is expensive with ~28 numeric features even in kernel mode. Budget real
compute time for a full run, or narrow `X` to a smaller column subset first.

In [ ]:
try:
    from XPER.compute.Performance import ModelPerformance
    HAS_XPER = True
except ImportError:
    HAS_XPER = False
    print("XPER not installed — pip install XPER, then re-run this cell")


class _XPERModelAdapter:
    """XPER calls model.predict_proba(X) directly; this routes that through our
    module.predict_proba(model, X) contract so it works regardless of what
    fit() returned."""

    def __init__(self, model, module):
        self._model, self._module = model, module

    def predict_proba(self, X):
        return self._module.predict_proba(self._model, X)


def compute_xper(model, module, X_train, y_train, X_test, y_test, metric="AUC", cfp=None, cfn=None):
    if not HAS_XPER:
        return None
    adapter = _XPERModelAdapter(model, module)
    xper = ModelPerformance(X_train, y_train, X_test, y_test, adapter)
    performance = xper.evaluate([metric], CFP=cfp, CFN=cfn)
    phi, phi_i_j = xper.calculate_XPER_values([metric], CFP=cfp, CFN=cfn, kernel=True)
    return {"performance": performance, "phi": phi, "phi_i_j": phi_i_j}

## Permutation importance — cheap cross-check

In [ ]:
def permutation_importance_report(model, module, X, y, n_repeats=10):
    return manual_permutation_importance(model, module, X, y, n_repeats=n_repeats)

## Run across available models

Loops over `available_models()` — a model with no `models/<name>_model.py` yet is
skipped, not errored on.

In [ ]:
results = {}

for name, module in available_models().items():
    print(f"\n=== {name} ===")
    model = module.fit(X_train, y_train)
    model_type = MODEL_EXPLAINER_TYPE[name]

    explainer, method = get_shap_explainer(model_type, model, module, X_train)
    print(f"  SHAP method: {method}")
    test_sample = X_test.sample(min(200, len(X_test)), random_state=42)
    shap_values = explainer(test_sample) if model_type != "kernel" else explainer.shap_values(test_sample)

    probs_test = module.predict_proba(model, X_test)[:, 1]
    auc = roc_auc_score(y_test, probs_test)
    xper_auc = compute_xper(model, module, X_train, y_train, X_test, y_test, metric="AUC")
    perm_imp = permutation_importance_report(model, module, X_test, y_test, n_repeats=5)

    results[name] = {
        "auc": auc,
        "shap_values": shap_values,
        "xper_auc": xper_auc,
        "permutation_importance": perm_imp,
    }
    print(f"  AUC: {auc:.4f}")

if not results:
    print("No models ready yet — drop a models/<name>_model.py file in and re-run.")

## Global comparison — mean |SHAP| per feature, across models

In [ ]:
if results:
    fig, ax = plt.subplots(figsize=(8, 5))
    for name, r in results.items():
        sv = r["shap_values"]
        vals = sv.values if hasattr(sv, "values") else np.asarray(sv)
        mean_abs = np.abs(vals).mean(axis=0)
        ax.barh(FEATURES, mean_abs, alpha=0.5, label=name)
    ax.set_xlabel("mean |SHAP value|")
    ax.legend()
    plt.tight_layout()
    plt.show()

## Smoke test — remove once real models are in `models/`

Validates the pipeline above actually runs against the real feature schema, using
a throwaway logistic regression on a small sample. Not a real submission.

In [ ]:
from sklearn.linear_model import LogisticRegression

class _SmokeTestModule:
    _medians = None  # fixed at fit time so a later all-NaN batch (e.g. a masked coalition) still fills

    @staticmethod
    def fit(X, y):
        Xn = X.select_dtypes("number")
        _SmokeTestModule._medians = Xn.median().fillna(0)
        return LogisticRegression(max_iter=200).fit(Xn.fillna(_SmokeTestModule._medians), y)

    @staticmethod
    def predict_proba(model, X):
        Xn = X.select_dtypes("number").fillna(_SmokeTestModule._medians)
        return model.predict_proba(Xn)

if not results:
    print("Running a throwaway smoke test — NOT a real model, just checking the harness works end to end.")
    sample = train_df.sample(20_000, random_state=42)
    Xs, ys = get_X_y(sample)
    smoke_model = _SmokeTestModule.fit(Xs, ys)
    smoke_test_sample = X_test.sample(2_000, random_state=42)
    smoke_probs = _SmokeTestModule.predict_proba(smoke_model, smoke_test_sample)
    print("Smoke test predict_proba shape:", smoke_probs.shape, "— harness is wired correctly.")

    if HAS_XPER:
        # kernel-based XPER's runtime scales with feature count, not row count — cut to a
        # handful of columns and fit a dedicated small model on just those, so this stays
        # a fast plumbing check (a model fit on all features would reject a 5-column input)
        xper_cols = FEATURES[:5]
        xper_train_sample = Xs[xper_cols]
        xper_smoke_model = _SmokeTestModule.fit(xper_train_sample, ys)
        xper_test_sample = X_test.sample(50, random_state=42)[xper_cols]
        xper_smoke = compute_xper(
            xper_smoke_model, _SmokeTestModule, xper_train_sample, ys,
            xper_test_sample, y_test.loc[xper_test_sample.index],
            metric="AUC",
        )
        print("XPER smoke test phi shape:", xper_smoke["phi"].shape, "— XPER call succeeded.")